# Lab 9 - Data Cleaning for Apple Brand Monitor

This notebook demonstrates the full Lab 9 cleaning workflow for the Apple/iPhone social brand monitoring dataset built in Lab 8.

Cleaning is performed on the Apple/iPhone project subset rather than on every mixed raw source record, because the project domain is brand monitoring for Apple and iPhone mentions.


In [14]:
from pathlib import Path
import logging
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analytics.regex_ops import (
    detect_invalid_date_formats,
    detect_invalid_language_codes,
    extract_numeric_values_from_text,
    flag_short_overviews,
)
from src.analytics.explorer import filter_apple_mentions
from src.cleaning.clean_pipeline import run_cleaning_pipeline
from src.cleaning.deduplicator import (
    count_duplicates,
    drop_duplicate_ids,
    drop_duplicate_title_date_pairs,
    remove_exact_duplicates,
)
from src.cleaning.missing_handler import handle_missing_values, report_missing
from src.cleaning.string_cleaner import clean_brand_strings
from src.cleaning.type_converter import convert_brand_types, memory_report
from src.cleaning.validator import validate_brand_dataset
from src.utils.logger import get_logger

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = get_logger("lab9_data_cleaning")

RAW_CSV_PATH = PROJECT_ROOT / "data" / "raw" / "csv" / "raw_brand_mentions.csv"
CLEANED_DIR = PROJECT_ROOT / "data" / "processed" / "cleaned"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
MISSING_REPORT_PATH = CLEANED_DIR / "missing_report.csv"
CLEANED_DATA_PATH = CLEANED_DIR / "cleaned_data.csv"


In [15]:
raw_df = pd.read_csv(RAW_CSV_PATH)
logger.info("Loaded %s rows and %s columns from %s", len(raw_df), len(raw_df.columns), RAW_CSV_PATH)
project_df = filter_apple_mentions(raw_df, keyword="apple")
logger.info("Restricted notebook cleaning data to %s Apple/iPhone rows", len(project_df))
project_df.head()

INFO: Loaded 558 rows and 37 columns from c:\Users\topic\OneDrive\Desktop\PROJECTS\Projekat za unstructured data\social_media_brand_monitor\data\raw\csv\raw_brand_mentions.csv
INFO: Apple mention filter complete | keyword=apple | rows_before=558 | rows_after=254
INFO: Restricted notebook cleaning data to 254 Apple/iPhone rows


,_id,source,author,content,description,document_type,extraction_timestamp,publishedAt,title,url,...,awards,best_picture,nominations,year,query_params.ajax,query_params.year,language,rating,content_hash,record_date
0,69ce9d979d3427d3654d3275,newsapi_Apple_page_1.json,bringatrailer,This Liquid Silver Metallic 2012 Mazda MX-5 Mi...,This Liquid Silver Metallic 2012 Mazda MX-5 Mi...,json,2026-04-25 20:39:13.637,2026-04-24T20:10:56Z,2012 Mazda MX-5 Miata Grand Touring PRHT 6-Spe...,https://bringatrailer.com/listing/2012-mazda-m...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,69ce9d979d3427d3654d3276,newsapi_Apple_page_2.json,Jusuf Hatic,Tim Cook hatte so einige Produkte in seiner Ze...,Nach über 15 Jahren hört Tim Cook als Apple-CE...,json,2026-04-25 20:39:13.673,2026-04-24T19:59:00Z,News: Nach Rücktritt - »Mein erster großer Feh...,"https://www.gamestar.de/artikel/,3452032.html",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,69ce9d979d3427d3654d3277,newsapi_Apple_page_3.json,Alex Griffing,"Todd Blanche, Deputy Attorney General of the U...",“To the extent the money encouraged or sustain...,json,2026-04-25 20:39:13.688,2026-04-24T19:32:40Z,‘Certainly Disreputable’: WSJ Editorial Board ...,https://www.mediaite.com/media/news/certainly-...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,69ce9d979d3427d3654d3278,sample.csv,Alex Brown,NaN,NaN,csv,2026-04-25 20:39:14.090,NaN,Apple doesnt lawsuit,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,69ce9d979d3427d3654d3279,sample.xml,Alex Brown,NaN,NaN,xml,2026-04-25 20:39:14.100,NaN,Apple faces lawsuit,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Missing Value Analysis


In [16]:
missing_report = report_missing(project_df)
missing_report.to_csv(MISSING_REPORT_PATH, index=False)
logger.info("Saved missing-value report to %s", MISSING_REPORT_PATH)
missing_report.head(15)

INFO: Generated missing-value report for 37 columns.
INFO: Saved missing-value report to c:\Users\topic\OneDrive\Desktop\PROJECTS\Projekat za unstructured data\social_media_brand_monitor\data\processed\cleaned\missing_report.csv


,column,dtype,missing_count,missing_ratio,non_missing_count
0,Author,str,254,1.0,0
1,CreationDate,str,254,1.0,0
2,Creator,str,254,1.0,0
3,Keywords,float64,254,1.0,0
4,ModDate,str,254,1.0,0
5,Producer,str,254,1.0,0
6,Subject,str,254,1.0,0
7,Title,str,254,1.0,0
8,Trapped,object,254,1.0,0
9,awards,float64,254,1.0,0


In [17]:
missing_clean_df = handle_missing_values(
    project_df,
    critical_columns=["_id", "title"],
    text_columns=["author", "description", "content", "title", "source", "document_type", "type"],
    zero_as_missing_columns=["rating", "price", "content_length"],
    numeric_columns=["rating", "price", "content_length", "page", "page_number", "query_params.year"],
    high_missing_threshold=0.85,
    protected_columns=["_id", "title", "source", "document_type", "type", "publishedAt", "date", "url", "language", "rating", "content_hash", "record_date"],
)
missing_clean_df.isna().sum().sort_values(ascending=False).head(15)

INFO: Dropped 0 rows with missing critical identifiers: ['_id', 'title']
INFO: Replaced unrealistic zeros with NaN in columns: ['rating', 'price']
INFO: Dropped 20 columns above missing threshold 0.85: ['Author', 'CreationDate', 'Creator', 'Keywords', 'ModDate', 'Producer', 'Subject', 'Title', 'Trapped', 'page', 'file_name', 'text', 'page_number', 'price', 'awards', 'best_picture', 'nominations', 'year', 'query_params.ajax', 'query_params.year']
INFO: Filled missing text values in 7 columns.
INFO: Filled missing numeric values with medians in 1 columns.
INFO: Completed missing-value handling. Shape: (254, 17)


content_hash            254
language                211
record_date             198
date                    193
urlToImage               76
url                      61
publishedAt              61
_id                       0
source                    0
title                     0
extraction_timestamp      0
document_type             0
description               0
author                    0
content                   0
dtype: int64

`price` is treated as non-essential in this project. If the Apple/iPhone subset contains no usable numeric values in that column, it is dropped instead of imputed because filling a completely empty numeric field would invent data.


## String Cleaning


In [18]:
string_clean_df = clean_brand_strings(missing_clean_df.copy())
preview_columns = [column for column in ["title", "language", "overview", "mention_date", "mention_year"] if column in string_clean_df.columns]
string_clean_df[preview_columns].head(10)

INFO: Normalized text formatting in columns: ['_id', 'source', 'author', 'content', 'description', 'document_type', 'extraction_timestamp', 'publishedAt', 'title', 'url', 'urlToImage', 'date', 'type', 'language', 'content_hash', 'record_date']
INFO: Cleaned title column: title
INFO: Normalized language codes in column: language
INFO: Created cleaned overview column overview from ['description', 'content']
INFO: Sanitized URL values in column: url
INFO: Created mention_date from source columns ['publishedAt', 'date', 'record_date']
INFO: Extracted year column mention_year from mention_date
INFO: Completed string cleaning. Shape: (254, 20)


,title,language,overview,mention_date,mention_year
0,2012 Mazda MX-5 Miata Grand Touring PRHT 6-Spe...,<NA>,This Liquid Silver Metallic 2012 Mazda MX-5 Mi...,2026-04-24,2026
1,News: Nach Rücktritt - »Mein erster großer Feh...,<NA>,Nach über 15 Jahren hört Tim Cook als Apple-CE...,2026-04-24,2026
2,‘Certainly Disreputable’: WSJ Editorial Board ...,<NA>,“To the extent the money encouraged or sustain...,2026-04-24,2026
3,Apple doesnt lawsuit,<NA>,Unknown,2026-03-02,2026
4,Apple faces lawsuit,<NA>,Unknown,2026-03-03,2026
15,AirPods daily use impressions,es,Unknown,2026-04-16,2026
17,Battlefield rare earths: How the U.S. lost to ...,<NA>,"At one point in history, one U.S. company mono...",2026-04-24,2026
18,"WiFi 6, WiFi 6E y WiFi 7: qué significan los n...",<NA>,Tu router dice «WiFi 6». Tu nuevo móvil presum...,2026-04-24,2026
19,Ads Are Coming to Apple Maps This Summer: Here...,<NA>,Apple is planning to start showing ads in the ...,2026-04-24,2026
20,Tertulia de Cuesta: de la caza de la UCO a Par...,<NA>,Miguel Ángel Pérez y Teresa Gómez analizan con...,2026-04-24,2026


## Regex Cleaning and Validation Flags


In [19]:
regex_df = extract_numeric_values_from_text(string_clean_df, "overview")
regex_df = flag_short_overviews(regex_df, column="overview", min_length=40)
invalid_published_dates = detect_invalid_date_formats(string_clean_df, column="publishedAt")
invalid_language_codes = detect_invalid_language_codes(string_clean_df, column="language")

print("Invalid publishedAt strings:", invalid_published_dates)
print("Invalid language codes:", invalid_language_codes)
regex_df[[column for column in ["overview", "overview_numeric_value", "overview_is_too_short"] if column in regex_df.columns]].head(10)

INFO: Numeric extraction completed | source_column=overview | output_column=overview_numeric_value
INFO: Short-overview scan completed | column=overview | min_length=40 | flagged=63
INFO: Invalid-date scan completed | column=publishedAt | invalid_count=0
INFO: Invalid-language scan completed | column=language | invalid_count=0


Invalid publishedAt strings: 0
Invalid language codes: 0


,overview,overview_numeric_value,overview_is_too_short
0,This Liquid Silver Metallic 2012 Mazda MX-5 Mi...,2012,False
1,Nach über 15 Jahren hört Tim Cook als Apple-CE...,15,False
2,“To the extent the money encouraged or sustain...,<NA>,False
3,Unknown,<NA>,True
4,Unknown,<NA>,True
15,Unknown,<NA>,True
17,"At one point in history, one U.S. company mono...",<NA>,False
18,Tu router dice «WiFi 6». Tu nuevo móvil presum...,6,False
19,Apple is planning to start showing ads in the ...,26.5,False
20,Miguel Ángel Pérez y Teresa Gómez analizan con...,<NA>,False


## Deduplication


In [20]:
dedup_df = string_clean_df.copy()
print("Rows before deduplication:", len(dedup_df))

dedup_df = remove_exact_duplicates(dedup_df)
print("Rows after exact duplicate removal:", len(dedup_df))

duplicate_id_count = count_duplicates(dedup_df, "_id")
print("Duplicate _id values before ID cleanup:", duplicate_id_count)

dedup_df = drop_duplicate_ids(dedup_df)
print("Rows after duplicate ID removal:", len(dedup_df))

dedup_df = drop_duplicate_title_date_pairs(dedup_df)
print("Rows after duplicate title/date removal:", len(dedup_df))

INFO: Removed 0 exact duplicate rows.
INFO: Counted 74 duplicate values in column _id
INFO: Removed 74 duplicate rows using identifier column _id
INFO: Removed 19 duplicate rows using identifier column url
INFO: Removed 0 duplicate rows using identifier column content_hash
INFO: Removed 6 duplicate title/date rows using columns title and mention_date


Rows before deduplication: 254
Rows after exact duplicate removal: 254
Duplicate _id values before ID cleanup: 74
Rows after duplicate ID removal: 161
Rows after duplicate title/date removal: 155


## Type Conversion


In [21]:
before_types_df = dedup_df.copy()
typed_df = convert_brand_types(dedup_df)
typed_df.dtypes.sort_index()

INFO: Converted datetime columns: ['publishedAt', 'date', 'mention_date', 'record_date', 'extraction_timestamp']
INFO: Converted numeric columns | ints=['mention_year'] | floats=['rating']
INFO: Converted category columns: ['source', 'document_type', 'type', 'language']
INFO: Completed type conversion. Shape: (155, 20)


_id                                  string
author                               string
content                              string
content_hash                         string
date                    datetime64[us, UTC]
description                          string
document_type                      category
extraction_timestamp    datetime64[us, UTC]
language                           category
mention_date            datetime64[us, UTC]
mention_year                          Int64
overview                             string
publishedAt             datetime64[us, UTC]
rating                              float32
record_date             datetime64[us, UTC]
source                             category
title                                string
type                               category
url                                  string
urlToImage                           string
dtype: object

In [22]:
memory_report(before_types_df, typed_df)

INFO: Memory report created | before_bytes=459150 | after_bytes=362187 | saved_bytes=96963


,stage,memory_bytes,memory_mb
0,before,459150,0.437880
1,after,362187,0.345408
2,saved,96963,0.092471


## Validation


In [23]:
validated_df = validate_brand_dataset(typed_df)
print("Validation checks passed for the cleaned Apple dataset.")
validated_df.head()

INFO: Invalid-date scan found 0 issues because publishedAt is already datetime.
INFO: Invalid-language scan completed | column=language | invalid_count=0
INFO: Validation completed successfully. Shape: (155, 20)


Validation checks passed for the cleaned Apple dataset.


,_id,source,author,content,description,document_type,extraction_timestamp,publishedAt,title,url,urlToImage,date,type,language,rating,content_hash,record_date,overview,mention_date,mention_year
0,69ce9d979d3427d3654d3275,newsapi_Apple_page_1.json,bringatrailer,This Liquid Silver Metallic 2012 Mazda MX-5 Mi...,This Liquid Silver Metallic 2012 Mazda MX-5 Mi...,json,2026-04-25 20:39:13.637000+00:00,2026-04-24 20:10:56+00:00,2012 Mazda MX-5 Miata Grand Touring PRHT 6-Spe...,https://bringatrailer.com/listing/2012-mazda-m...,https://bringatrailer.com/wp-content/uploads/2...,NaT,Unknown,<NA>,3.0,<NA>,NaT,This Liquid Silver Metallic 2012 Mazda MX-5 Mi...,2026-04-24 00:00:00+00:00,2026
1,69ce9d979d3427d3654d3276,newsapi_Apple_page_2.json,Jusuf Hatic,Tim Cook hatte so einige Produkte in seiner Ze...,Nach über 15 Jahren hört Tim Cook als Apple-CE...,json,2026-04-25 20:39:13.673000+00:00,2026-04-24 19:59:00+00:00,News: Nach Rücktritt - »Mein erster großer Feh...,"https://www.gamestar.de/artikel/,3452032.html",https://images.cgames.de/images/gamestar/293/a...,NaT,Unknown,<NA>,3.0,<NA>,NaT,Nach über 15 Jahren hört Tim Cook als Apple-CE...,2026-04-24 00:00:00+00:00,2026
2,69ce9d979d3427d3654d3277,newsapi_Apple_page_3.json,Alex Griffing,"Todd Blanche, Deputy Attorney General of the U...",“To the extent the money encouraged or sustain...,json,2026-04-25 20:39:13.688000+00:00,2026-04-24 19:32:40+00:00,‘Certainly Disreputable’: WSJ Editorial Board ...,https://www.mediaite.com/media/news/certainly-...,https://www.mediaite.com/wp-content/uploads/20...,NaT,Unknown,<NA>,3.0,<NA>,NaT,“To the extent the money encouraged or sustain...,2026-04-24 00:00:00+00:00,2026
3,69ce9d979d3427d3654d3278,sample.csv,Alex Brown,Unknown,Unknown,csv,2026-04-25 20:39:14.090000+00:00,NaT,Apple doesnt lawsuit,<NA>,<NA>,2026-03-02 00:00:00+00:00,Unknown,<NA>,3.0,<NA>,NaT,Unknown,2026-03-02 00:00:00+00:00,2026
4,69ce9d979d3427d3654d3279,sample.xml,Alex Brown,Unknown,Unknown,xml,2026-04-25 20:39:14.100000+00:00,NaT,Apple faces lawsuit,<NA>,<NA>,2026-03-03 00:00:00+00:00,Unknown,<NA>,3.0,<NA>,NaT,Unknown,2026-03-03 00:00:00+00:00,2026


## Reusable Cleaning Pipeline


In [24]:
final_clean_df = run_cleaning_pipeline(project_df, output_path=CLEANED_DATA_PATH)
logger.info("Saved cleaned dataset to %s", CLEANED_DATA_PATH)
final_clean_df.head()

INFO: Cleaning pipeline started | rows=254 | columns=37
INFO: Apple mention filter complete | keyword=apple | rows_before=254 | rows_after=254
INFO: Restricted cleaning dataset to Apple/iPhone project scope | rows=254 | columns=37
INFO: Generated missing-value report for 37 columns.
INFO: Missing-value report saved | path=data\processed\cleaned\missing_report.csv
INFO: Dropped 0 rows with missing critical identifiers: ['_id', 'title']
INFO: Replaced unrealistic zeros with NaN in columns: ['rating', 'price']
INFO: Dropped 20 columns above missing threshold 0.85: ['Author', 'CreationDate', 'Creator', 'Keywords', 'ModDate', 'Producer', 'Subject', 'Title', 'Trapped', 'page', 'file_name', 'text', 'page_number', 'price', 'awards', 'best_picture', 'nominations', 'year', 'query_params.ajax', 'query_params.year']
INFO: Filled missing text values in 7 columns.
INFO: Filled missing numeric values with medians in 1 columns.
INFO: Completed missing-value handling. Shape: (254, 17)
INFO: Normalized 

,_id,source,author,content,description,document_type,extraction_timestamp,publishedAt,title,url,urlToImage,date,type,language,rating,content_hash,record_date,overview,mention_date,mention_year
0,69ce9d979d3427d3654d3275,newsapi_Apple_page_1.json,bringatrailer,This Liquid Silver Metallic 2012 Mazda MX-5 Mi...,This Liquid Silver Metallic 2012 Mazda MX-5 Mi...,json,2026-04-25 20:39:13.637000+00:00,2026-04-24 20:10:56+00:00,2012 Mazda MX-5 Miata Grand Touring PRHT 6-Spe...,https://bringatrailer.com/listing/2012-mazda-m...,https://bringatrailer.com/wp-content/uploads/2...,NaT,Unknown,<NA>,3.0,<NA>,NaT,This Liquid Silver Metallic 2012 Mazda MX-5 Mi...,2026-04-24 00:00:00+00:00,2026
1,69ce9d979d3427d3654d3276,newsapi_Apple_page_2.json,Jusuf Hatic,Tim Cook hatte so einige Produkte in seiner Ze...,Nach über 15 Jahren hört Tim Cook als Apple-CE...,json,2026-04-25 20:39:13.673000+00:00,2026-04-24 19:59:00+00:00,News: Nach Rücktritt - »Mein erster großer Feh...,"https://www.gamestar.de/artikel/,3452032.html",https://images.cgames.de/images/gamestar/293/a...,NaT,Unknown,<NA>,3.0,<NA>,NaT,Nach über 15 Jahren hört Tim Cook als Apple-CE...,2026-04-24 00:00:00+00:00,2026
2,69ce9d979d3427d3654d3277,newsapi_Apple_page_3.json,Alex Griffing,"Todd Blanche, Deputy Attorney General of the U...",“To the extent the money encouraged or sustain...,json,2026-04-25 20:39:13.688000+00:00,2026-04-24 19:32:40+00:00,‘Certainly Disreputable’: WSJ Editorial Board ...,https://www.mediaite.com/media/news/certainly-...,https://www.mediaite.com/wp-content/uploads/20...,NaT,Unknown,<NA>,3.0,<NA>,NaT,“To the extent the money encouraged or sustain...,2026-04-24 00:00:00+00:00,2026
3,69ce9d979d3427d3654d3278,sample.csv,Alex Brown,Unknown,Unknown,csv,2026-04-25 20:39:14.090000+00:00,NaT,Apple doesnt lawsuit,<NA>,<NA>,2026-03-02 00:00:00+00:00,Unknown,<NA>,3.0,<NA>,NaT,Unknown,2026-03-02 00:00:00+00:00,2026
4,69ce9d979d3427d3654d3279,sample.xml,Alex Brown,Unknown,Unknown,xml,2026-04-25 20:39:14.100000+00:00,NaT,Apple faces lawsuit,<NA>,<NA>,2026-03-03 00:00:00+00:00,Unknown,<NA>,3.0,<NA>,NaT,Unknown,2026-03-03 00:00:00+00:00,2026
